# Reranking Sanity Check

Picks up where `retrieval.ipynb` left off. That notebook exposed a real failure mode of pure bi-encoder retrieval: the query *"reinforcement learning from human feedback for language model alignment"* returned a confident-looking top-1 (cosine 0.62) for a multilingual model merging paper — thematically off.

The cross-encoder reranker is the spec'd fallback for exactly this case. From `PLAN.md`:

> *"As fallbacks, we use a pretrained cross-encoder reranker when abstracts are missing or low-quality..."*

Where the bi-encoder embeds query and passage independently and scores them by cosine, the cross-encoder scores `(query, passage)` jointly with full self-attention across both — slower per pair, but markedly better at separating semantically close-but-off-topic candidates from true matches.

This notebook:
1. Reproduces the ambiguous RLHF query from `retrieval.ipynb`.
2. Applies `rerank_passages()` to its results and compares before vs. after.
3. Demonstrates the `should_rerank()` confidence trigger (PLAN.md fallback semantics).
4. Wires retrieve → trigger → rerank into a single end-to-end function.

## Setup

Same load pattern as `retrieval.ipynb` plus the cross-encoder. The cross-encoder model (`cross-encoder/ms-marco-MiniLM-L-6-v2`, ~90 MB) downloads to the HuggingFace cache on first run.

In [1]:
import json
import os
import sys

import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
import config
from src.encoder import load_model
from src.retrieval import retrieve
from src.reranker import load_reranker, rerank_passages, should_rerank

EMBEDDINGS_DIR = config.EMBEDDINGS_DIR

abstracts = np.load(os.path.join(EMBEDDINGS_DIR, "abstracts.npy"))
chunks    = np.load(os.path.join(EMBEDDINGS_DIR, "chunks.npy"))
captions  = np.load(os.path.join(EMBEDDINGS_DIR, "captions.npy"))

with open(os.path.join(EMBEDDINGS_DIR, "index.json")) as f:
    index = json.load(f)

encoder_model_name = index.get("encoder_model", config.ENCODER_MODEL_NAME)
encoder = load_model(encoder_model_name)
reranker = load_reranker()

print(f"Bi-encoder    : {encoder_model_name}")
print(f"Cross-encoder : cross-encoder/ms-marco-MiniLM-L-6-v2")
print(f"Corpus        : {abstracts.shape[0]} papers, {chunks.shape[0]} chunks, {captions.shape[0]} captions")

Bi-encoder    : all-mpnet-base-v2
Cross-encoder : cross-encoder/ms-marco-MiniLM-L-6-v2
Corpus        : 273 papers, 7634 chunks, 2686 captions



Loading weights: 100%|##########| 199/199 [00:00<00:00, 19125.74it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

Loading weights: 100%|##########| 105/105 [00:00<00:00, 18538.56it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 1. Reproduce the ambiguous RLHF query

Same query, same `K`, same encoder as `retrieval.ipynb` cell 4. Retrieve a wider candidate pool (`K=10`) than the spec'd display count of 5 — a deeper pool gives the cross-encoder more material to rerank from. See `config.N_RETRIEVAL_RESULTS` (20) and `config.N_RERANK_RESULTS` (5).

In [2]:
QUERY = "reinforcement learning from human feedback for language model alignment"
K = 10

result = retrieve(
    QUERY,
    abstracts,
    chunks,
    captions,
    index,
    k=K,
    model=encoder,
    top_chunks_per_paper=3,
)

print(f"Query: {QUERY!r}\n")
print(f"Top {K} papers by abstract cosine (bi-encoder):")
for rank, (pid, score) in enumerate(zip(result['paper_ids'], result['scores']), 1):
    title = next((c['paper_title'] for c in index['chunks'] if c['paper_id'] == pid), pid)
    print(f"  {rank:>2}. [{score:.4f}]  {title[:90]}")

print(f"\nReturned {len(result['passages'])} candidate passages for reranking.")

Query: 'reinforcement learning from human feedback for language model alignment'

Top 10 papers by abstract cosine (bi-encoder):
   1. [0.6177]  Merge and Conquer: Instructing Multilingual Models by Adding Target Language Weights
   2. [0.5274]  Visualization of Machine Learning Models through Their Spatial and Temporal Listeners
   3. [0.5176]  Understanding Teacher Revisions of Large Language Model-Generated Feedback
   4. [0.4957]  ATLAS-RTC: Closing the Loop on LLM Agent Output with Token-Level Runtime Control
   5. [0.4924]  Rethinking Easy-to-Hard: Limits of Curriculum Learning in Post-Training for Deductive Reas
   6. [0.4878]  Mitigating Hallucination on Hallucination in RAG via Ensemble Voting
   7. [0.4874]  Conditional Factuality Controlled LLMs with Generalization Certificates via Conformal Samp
   8. [0.4768]  The Model Says Walk: How Surface Heuristics Override Implicit Constraints in LLM Reasoning
   9. [0.4745]  Improving Attributed Long-form Question Answering with Int


Batches: 100%|##########| 1/1 [00:00<00:00,  7.37it/s]


## 2. Apply the cross-encoder reranker

`rerank_passages()` consumes the dict-form passages from `retrieve()` directly — no shape conversion needed. The cross-encoder score replaces the bi-encoder cosine in each dict so downstream consumers (LLM context builder) treat reranked and non-reranked results uniformly.

In [3]:
before = result["passages"][:5]
after  = rerank_passages(QUERY, result["passages"], model=reranker)[:5]

def show(label, passages):
    print(f"\n=== {label} ===")
    for i, p in enumerate(passages, 1):
        snippet = p['text'][:100].replace('\n', ' ').strip()
        print(f"  {i}. [{p['score']:>7.4f}] {p['paper_title'][:70]}")
        print(f"             / {p['heading'][:60]}")
        print(f"             {snippet}...")

show("BEFORE — bi-encoder cosine", before)
show("AFTER  — cross-encoder relevance", after)

before_ids = [p['paper_id'] for p in before]
after_ids  = [p['paper_id'] for p in after]
moved      = [pid for pid in after_ids if pid not in before_ids]
if moved:
    print(f"\nReranker promoted {len(moved)} paper(s) into the top-5 that the bi-encoder ranked lower:")
    for pid in moved:
        title = next((c['paper_title'] for c in index['chunks'] if c['paper_id'] == pid), pid)
        print(f"   + {title[:90]}")
else:
    print("\nReranker kept the same top-5 papers (only re-ordered within).")


=== BEFORE — bi-encoder cosine ===
  1. [ 0.5860] Merge and Conquer: Instructing Multilingual Models by Adding Target La
             / Acknowledgments
             This work has been funded by the Spanish Ministry of Science, Innovation, and Universities (Project...
  2. [ 0.5835] Merge and Conquer: Instructing Multilingual Models by Adding Target La
             / 4.  Experimental Setup
             To thoroughly evaluate our hypotheses, we conducted experiments across multiple languages (Basque, G...
  3. [ 0.5687] Merge and Conquer: Instructing Multilingual Models by Adding Target La
             / 7.  Limitations
             Despite the promising results, this work has several limitations. Our study is restricted to a limit...
  4. [ 0.5585] Understanding Teacher Revisions of Large Language Model-Generated Feed
             / 3.1Dataset and Preprocessing
             The platform used two large language models during its development. Early versions relied on GPT-4 (...
  5. [ 0.

## 3. Confidence trigger — `should_rerank()`

Reranking every query is wasteful (the cross-encoder is ~10× slower per pair than a cosine lookup). Per PLAN.md it's a **fallback**, fired only when the bi-encoder's output looks unreliable. Two signals approximate "unreliable":

1. **Top-1 score below a floor** — no candidate looks strong in absolute terms.
2. **Tight top-1 vs top-K spread** — candidates are bunched together, the bi-encoder cannot separate them, and a joint cross-encoder score is more likely to break the tie.

Either condition triggers a rerank pass.

In [ ]:
demo_queries = [
    # Top-1 cosine looks confident in isolation, but top-K are bunched
    # together — the bi-encoder can't separate the retrieved papers.
    QUERY,
    # Many diffusion papers in the corpus score similarly — tight spread
    # triggers reranking for within-group refinement.
    "diffusion models for image generation",
    # Out-of-corpus query — top-1 itself is weak, triggers the score floor.
    "a topic the corpus has nothing to say about — quantum cryptography in 1972",
]

for q in demo_queries:
    r = retrieve(q, abstracts, chunks, captions, index, k=5, model=encoder)
    scores = r["scores"]
    top1, topK = scores[0], scores[-1]
    spread = top1 - topK
    fire = should_rerank(scores)
    print(f"Query: {q[:70]!r}")
    print(f"   top-1={top1:.4f}  top-K={topK:.4f}  spread={spread:.4f}  →  rerank? {fire}\n")

## 4. End-to-end: retrieve with auto-reranking fallback

Single function that ties retrieve → trigger → rerank. This is the shape `app.py` will call from `POST /api/query` once Kevin Pei wires up `src/llm.py`.

In [5]:
def retrieve_with_fallback(query, k=10, top_n=5):
    """Bi-encoder retrieve, optionally reranked by cross-encoder when scores look weak."""
    r = retrieve(query, abstracts, chunks, captions, index,
                 k=k, model=encoder, top_chunks_per_paper=3)
    fired = should_rerank(r["scores"])
    if fired:
        r["passages"] = rerank_passages(query, r["passages"], model=reranker)
    r["reranked"] = fired
    r["passages"] = r["passages"][:top_n]
    return r

for q in [QUERY, "diffusion models for image generation"]:
    r = retrieve_with_fallback(q)
    print(f"\nQuery: {q!r}")
    print(f"   reranked: {r['reranked']}")
    for i, p in enumerate(r['passages'], 1):
        print(f"   {i}. [{p['score']:>7.4f}] {p['paper_title'][:70]} / {p['heading'][:40]}")


Query: 'reinforcement learning from human feedback for language model alignment'
   reranked: False
   1. [ 0.5860] Merge and Conquer: Instructing Multilingual Models by Adding Target La / Acknowledgments
   2. [ 0.5835] Merge and Conquer: Instructing Multilingual Models by Adding Target La / 4.  Experimental Setup
   3. [ 0.5687] Merge and Conquer: Instructing Multilingual Models by Adding Target La / 7.  Limitations
   4. [ 0.5585] Understanding Teacher Revisions of Large Language Model-Generated Feed / 3.1Dataset and Preprocessing
   5. [ 0.5308] Improving Attributed Long-form Question Answering with Intent Awarenes / A.2The Use of Large Language Models (LLM

Query: 'diffusion models for image generation'
   reranked: False
   1. [ 0.6620] LDDMM stochastic interpolants: an application to domain uncertainty qu / 1.Introduction
   2. [ 0.6410] $R_\text{dm}$: Re-conceptualizing Distribution Matching as a Reward fo / 6Conclusion
   3. [ 0.6353] ImagenWorld: Stress-Testing Image Generat


Batches: 100%|##########| 1/1 [00:00<00:00, 90.11it/s]

Batches: 100%|##########| 1/1 [00:00<00:00, 94.72it/s]
